# PANAR-X — Full Reproducibility Pipeline (Colab)

This notebook re-runs the **complete** training and audit pipeline behind
*Stability-Constrained Regime-Aware Forecasting for Heterogeneous Panel
Time Series* (PANAR-X) on Colab GPU and verifies the regenerated
artifacts against the paper's numerical claims.

**Expected runtime: ~85-100 minutes on a free T4 GPU.** (~8 hours on CPU.)

## What this notebook does

| Stage | Output | T4 GPU time |
|---|---|---|
| 0. Setup, Drive mount, GPU check, dependency install | — | ~1 min |
| 1. Smoke-test against saved artifacts (no training) | `141/141 PASS` | ~5 sec |
| 2. *(Optional)* Snapshot original artifacts | `artifacts_original/` | ~1 sec |
| 3. Synthetic v1 baselines (M1, M1.5, M2 reference) | `artifacts/synthetic_v1_results.pkl` | ~5 min |
| 4. Synthetic Pareto-frontier sweep (45 runs) | `artifacts/synthetic_v1f_frontier.pkl` | ~30 min |
| 5. V-Dem 50-run sweep + baselines | `artifacts/vdem_v1h_frontier.pkl` | ~30 min |
| 6. V-Dem postanalysis (paired DM predictions) | `artifacts/vdem_v1h_postanalysis.pkl` | ~22 min |
| 7. Certified spectral-norm bound audit (App D.2) | `artifacts/certified_audit_results.pkl` | ~2 min |
| 8. Build paper Figure 1 | `paper/figure_diagnostic.{pdf,png}` | ~15 sec |
| 9. Verify regenerated artifacts vs paper numbers | `141/141 PASS` (or near-141) | ~5 sec |
| 10. Diff regenerated vs original artifacts | numerical drift report | ~5 sec |
| 11. Save everything to Drive | persistent output | ~30 sec |

## Important caveats

- **GPU non-determinism**: the verify script uses tight tolerances
  (`tol_abs=5e-6` on baseline MSEs). Re-running on Colab GPU typically
  introduces small numerical differences from non-deterministic CUDA
  matmul kernels. Expect some of the 141 checks to fail by epsilon
  margins. Read the diff in stage 10 to confirm the drift is noise,
  not structural.
- **Don't republish the regenerated tarball as `141/141 PASS`** — submit
  the original package with the saved artifacts. This notebook is for
  *verifying the pipeline reproduces*, not for replacing the artifacts.
- **Free Colab idle timeout**: ~90 minutes. Pipeline runtime is now
  longer than the free idle timeout, so use Colab Pro for the full
  retraining run, or skip the long stages and only run the audit.

## Before you start

1. **Runtime → Change runtime type → T4 GPU** (or any GPU runtime).
2. Upload `panar-x-reproducibility.tar.gz` (cell 0.3 will prompt).
3. Run cells top-to-bottom. Each is idempotent and can be re-run.

## 0. Setup

### 0.1 Mount Google Drive

Outputs from this notebook will be saved to `MyDrive/panar-x-replication/`.
A run-tagged subfolder is created so re-runs don't overwrite each other.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
DRIVE_OUTPUT = '/content/drive/MyDrive/panar-x-replication'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive output directory: {DRIVE_OUTPUT}')

# Subdirectory tagged with the start time so re-runs don't overwrite
RUN_TAG = time.strftime('%Y%m%d_%H%M%S')
RUN_DIR = f'{DRIVE_OUTPUT}/run_{RUN_TAG}'
os.makedirs(RUN_DIR, exist_ok=True)
print(f"This run's outputs will go to: {RUN_DIR}")

### 0.2 Confirm GPU

You should see a T4, A100, V100, or L4 here. If you see "False", go to
*Runtime → Change runtime type* and select a GPU.

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device name:    {torch.cuda.get_device_name(0)}')
    print(f'CUDA version:   {torch.version.cuda}')
    print(f'PyTorch:        {torch.__version__}')
else:
    print('WARNING: no GPU detected — pipeline will run on CPU (~8 hours)')

### 0.3 Upload `panar-x-reproducibility.tar.gz`

Run this cell, then click *Choose Files* and select the tarball from your
local machine. (~1.1 MB upload.)

In [ ]:
import os
TARBALL = 'panar-x-reproducibility.tar.gz'

if os.path.exists(TARBALL):
    print(f'Found existing {TARBALL} — skipping upload.')
    print(f'Size: {os.path.getsize(TARBALL):,} bytes')
else:
    from google.colab import files
    print('Choose panar-x-reproducibility.tar.gz from your local machine:')
    uploaded = files.upload()
    if TARBALL not in uploaded:
        # Accept any tarball name and rename
        for k in uploaded:
            if k.endswith('.tar.gz'):
                os.rename(k, TARBALL)
                print(f'Renamed {k} -> {TARBALL}')
                break

### 0.4 Extract and enter the package directory

The `%cd` magic (not `!cd`) is required so the working directory persists
across subsequent cells.

In [ ]:
import os, subprocess

if not os.path.exists('panar-x-reproducibility'):
    subprocess.run(['tar', 'xzf', 'panar-x-reproducibility.tar.gz'], check=True)
    print('Extracted.')
else:
    print('Already extracted.')

%cd panar-x-reproducibility
print(f'\nCurrent directory: {os.getcwd()}')
print(f'\nTop-level contents:')
!ls -la

### 0.5 Install Python dependencies

Most are already in Colab. Should take ~30 seconds.

In [ ]:
!pip install -r requirements.txt --quiet 2>&1 | tail -5

## 1. Smoke test against saved artifacts

This step **does not retrain anything** — it loads the four pickles in
`artifacts/` (which ship with the package) and verifies that 141 paper-cited
numbers reproduce exactly. Should run in ~5 seconds.

If this fails, the package is corrupted or your Python environment has
unexpected library versions. Resolve before proceeding.

In [ ]:
import subprocess, sys

print('Running verify_paper_numbers.py against saved artifacts...\n')

with open('/tmp/smoke_test.log', 'w') as logf:
    proc = subprocess.Popen(
        ['python', '-u', 'verify_paper_numbers.py'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)
        logf.flush()
        sys.stdout.flush()
    proc.wait()

print(f'\n[Smoke test return code: {proc.returncode}]')

## 2. Snapshot original artifacts (recommended)

Each training stage will overwrite its corresponding pickle in
`artifacts/`. This step copies the originals to `artifacts_original/` so
you can diff after the run completes.

In [ ]:
import shutil, os

if os.path.exists('artifacts_original'):
    print('artifacts_original/ already exists — leaving alone.')
else:
    shutil.copytree('artifacts', 'artifacts_original')
    print('Snapshotted artifacts/ -> artifacts_original/')

print('\nFiles in artifacts_original/:')
!ls -la artifacts_original/

### Streaming subprocess helper

The next five stages each run a Python script for several seconds to
~30 minutes. To make sure progress shows up in the cell output as the
script runs (rather than appearing all at once when the script
finishes), the cell below defines a helper `run_stage()` that uses
`Popen` with line-buffered streaming.

This is the difference between an apparently-frozen cell and a cell that
shows live progress. Run this cell before any of the training stages.

In [ ]:
import subprocess, sys, time

def run_stage(stage_label, script_path, log_path):
    """Run a Python script with live output to the cell AND a log file.

    Uses Popen + line buffering so output appears in the cell as the
    script runs, not after it finishes. Critical on Colab where the
    naive subprocess.run + tee pattern double-buffers and the cell
    appears frozen for many minutes at a time.
    """
    print(f'{stage_label}')
    print('=' * 60)
    t0 = time.time()

    with open(log_path, 'w') as logf:
        proc = subprocess.Popen(
            ['python', '-u', script_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        try:
            for line in proc.stdout:
                print(line, end='')
                logf.write(line)
                logf.flush()
                sys.stdout.flush()
        except KeyboardInterrupt:
            proc.terminate()
            print('\n[Interrupted by user; process terminated.]')
            raise
        proc.wait()

    elapsed = time.time() - t0
    print(f'\n[{stage_label} elapsed: {elapsed/60:.1f} min, '
          f'return code: {proc.returncode}, log: {log_path}]')
    return proc.returncode

print('run_stage() defined.')

## 3. Synthetic v1 baselines

Trains the M1, M1.5, and M2 single-seed reference models on the
synthetic 3-regime panel. The Pareto-frontier sweep in stage 4 loads
this output to use the M1 and M1.5 baseline test-MSE values for gap
computation. Outputs `artifacts/synthetic_v1_results.pkl`.

**Expected runtime: ~5 minutes on T4 GPU, ~5 minutes on CPU.**

In [ ]:
run_stage(
    stage_label='Stage 3: Synthetic v1 baselines (M1, M1.5, M2 reference)',
    script_path='code/synthetic/synthetic_v1_baselines.py',
    log_path='/tmp/stage3.log',
)

## 4. Synthetic Pareto-frontier sweep (45 runs)

Trains M2 architecture across 9 cells × 5 seeds on the synthetic 3-regime
panel, sweeping enforcement modes (soft penalty, projection-per-epoch,
projection-per-batch) × disagreement penalty strengths.
Outputs `artifacts/synthetic_v1f_frontier.pkl`.

**Expected runtime: ~30 minutes on T4 GPU.**

In [ ]:
run_stage(
    stage_label='Stage 4: Synthetic Pareto-frontier sweep (45 runs)',
    script_path='code/synthetic/synthetic_v1f_sweep.py',
    log_path='/tmp/stage4.log',
)

## 5. V-Dem 50-run sweep + baselines

Trains: linear baselines (pooled, stratified), NN baselines (M1, M1.5),
and 50 M2 runs (10 cells × 5 seeds, with full audit constants per run).
Outputs `artifacts/vdem_v1h_frontier.pkl`.

**Expected runtime: ~30 minutes on T4 GPU.**

In [ ]:
run_stage(
    stage_label='Stage 5: V-Dem 50-run sweep + baselines',
    script_path='code/vdem/vdem_v1h_v3_cell.py',
    log_path='/tmp/stage5.log',
)

## 6. V-Dem postanalysis (paired DM predictions)

Retrains baselines and 5 M2 models from scratch to recover per-observation
predictions for the headline cell + all baselines. Used by the paired
Diebold-Mariano tests in §6.3 (Table 5.4) and Figure 1.
Outputs `artifacts/vdem_v1h_postanalysis.pkl`.

**Expected runtime: ~22 minutes on T4 GPU.** Most of this is the 5 M2
retrainings (~3.5 min/seed). The Jacobian-penalty backward pass doesn't
parallelize well, so A100 is only ~1.5× faster than T4 here.

In [ ]:
run_stage(
    stage_label='Stage 6: V-Dem postanalysis (paired DM predictions)',
    script_path='code/vdem/vdem_v1h_postanalysis.py',
    log_path='/tmp/stage6.log',
)

## 7. Certified spectral-norm bound audit (Appendix D.2)

Trains 3 vanilla-MSE M2 models, then computes per-regime gradient-probe
$\hat L^{(r)}_Y$ vs certified spectral-norm $\widetilde L^{(r)}_Y$ to
validate that the gradient probe is non-vacuous.
Outputs `artifacts/certified_audit_results.pkl`.

**Expected runtime: ~2 minutes on T4 GPU.**

In [ ]:
run_stage(
    stage_label='Stage 7: Certified spectral-norm bound audit (App D.2)',
    script_path='code/vdem/certified_bound_audit.py',
    log_path='/tmp/stage7.log',
)

## 8. Build paper Figure 1

Generates the two-row diagnostic+recovery figure from
`vdem_v1h_postanalysis.pkl`. Outputs `paper/figure_diagnostic.{pdf,png}`.

**Expected runtime: ~15 sec.**

In [ ]:
run_stage(
    stage_label='Stage 8: Build paper Figure 1',
    script_path='code/vdem/build_figure.py',
    log_path='/tmp/stage8.log',
)

# Display the regenerated figure inline
from IPython.display import Image
Image('paper/figure_diagnostic.png')

## 9. Verify regenerated artifacts vs paper numbers

This re-runs `verify_paper_numbers.py` against the **regenerated**
artifacts. On GPU you should expect a few `[FAIL]` entries with epsilon
differences (~1e-6) due to CUDA non-determinism — this is normal.

In [ ]:
import subprocess, sys

print('Running verify_paper_numbers.py against regenerated artifacts...\n')

with open('/tmp/verify_after.log', 'w') as logf:
    proc = subprocess.Popen(
        ['python', '-u', 'verify_paper_numbers.py'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)
        logf.flush()
        sys.stdout.flush()
    proc.wait()

# Extract pass/fail counts from the log
with open('/tmp/verify_after.log') as f:
    log = f.read()
pass_count = log.count('[PASS]')
fail_count = log.count('[FAIL]')
print(f'\n--- Summary ---')
print(f'PASS: {pass_count}')
print(f'FAIL: {fail_count}')

if fail_count > 0:
    print('\n--- Failed checks ---')
    for line in log.split('\n'):
        if '[FAIL]' in line:
            print(f'  {line}')
    print('\nSmall numerical differences (~1e-6) on GPU re-runs are expected')
    print('and harmless. Differences ~1e-4 or larger may indicate environment')
    print('drift; see stage 9 for quantitative drift analysis.')

## 10. Diff regenerated vs original artifacts

Quantifies the numerical drift between the original artifacts (shipped
with the package) and the artifacts you just regenerated. On GPU this
typically shows differences on the order of `1e-6` to `1e-4` for
test-MSE values, with audit constants slightly larger (~1e-3) because
they involve max-over-samples Jacobian probes.

If the drift is several orders of magnitude larger than that, something
is structurally different — investigate before submission.

In [ ]:
import pickle
import numpy as np
import os

print('=' * 70)
print('Diffing regenerated vs original artifacts')
print('=' * 70)

# --- Synthetic frontier ---
print('\n[1/3] synthetic_v1f_frontier.pkl\n')
with open('artifacts_original/synthetic_v1f_frontier.pkl', 'rb') as f:
    orig_s = pickle.load(f)
with open('artifacts/synthetic_v1f_frontier.pkl', 'rb') as f:
    new_s = pickle.load(f)

m15_drift = abs(orig_s['m15_test_mse'] - new_s['m15_test_mse'])
print(f"  M1.5 baseline:  orig={orig_s['m15_test_mse']:.6f}  "
      f"new={new_s['m15_test_mse']:.6f}  diff={m15_drift:.2e}")

# Per-cell test_mse mean drift
max_cell_drift = 0.0
for k in orig_s['results']:
    o_mses = np.array([r['test_mse'] for r in orig_s['results'][k]])
    n_mses = np.array([r['test_mse'] for r in new_s['results'][k]])
    drift = abs(o_mses.mean() - n_mses.mean())
    max_cell_drift = max(max_cell_drift, drift)
print(f'  Max per-cell test_mse mean drift across 9 cells: {max_cell_drift:.2e}')

# --- V-Dem frontier ---
print('\n[2/3] vdem_v1h_frontier.pkl\n')
with open('artifacts_original/vdem_v1h_frontier.pkl', 'rb') as f:
    orig_v = pickle.load(f)
with open('artifacts/vdem_v1h_frontier.pkl', 'rb') as f:
    new_v = pickle.load(f)

for k in ['RW', 'pooled_linear_M1.5', 'stratified_linear_M1.5', 'pooled_NN_M1.5']:
    o = orig_v['baselines'][k]
    n = new_v['baselines'][k]
    print(f'  {k:<28} orig={o:.6f}  new={n:.6f}  diff={abs(o-n):.2e}')

# Headline soft_only lam_M=0 audit constants drift
o_unc = orig_v['results'][('soft_only', 0.0)]
n_unc = new_v['results'][('soft_only', 0.0)]
for fld in ['L_eq_max', 'M', 'B_trans', 'test_mse']:
    o_v = np.mean([s[fld] for s in o_unc])
    n_v = np.mean([s[fld] for s in n_unc])
    print(f'  unc {fld:<14} orig={o_v:.4f}  new={n_v:.4f}  diff={abs(o_v-n_v):.2e}')

# --- Certified bound ---
print('\n[3/3] certified_audit_results.pkl\n')
with open('artifacts_original/certified_audit_results.pkl', 'rb') as f:
    orig_c = pickle.load(f)
with open('artifacts/certified_audit_results.pkl', 'rb') as f:
    new_c = pickle.load(f)

o_pmax = np.array([max(r['probe']) for r in orig_c['results']])
n_pmax = np.array([max(r['probe']) for r in new_c['results']])
o_cmax = np.array([max(r['cert']) for r in orig_c['results']])
n_cmax = np.array([max(r['cert']) for r in new_c['results']])
print(f'  probe max-over-regime mean: orig={o_pmax.mean():.4f}  '
      f'new={n_pmax.mean():.4f}  diff={abs(o_pmax.mean()-n_pmax.mean()):.2e}')
print(f'  cert  max-over-regime mean: orig={o_cmax.mean():.4f}  '
      f'new={n_cmax.mean():.4f}  diff={abs(o_cmax.mean()-n_cmax.mean()):.2e}')

print('\n' + '=' * 70)
print('Interpretation:')
print('  - drifts <= 1e-4: GPU noise, expected, harmless')
print('  - drifts ~ 1e-3:  borderline; expected for audit constants')
print('  - drifts >= 1e-2: investigate (could indicate seed mismatch or')
print('                    structural code drift)')
print('=' * 70)

## 11. Save outputs to Drive

Copies the regenerated artifacts, the figure, and the verify logs to the
Drive folder set up at the top of this notebook. Files saved:

- `artifacts/` — regenerated pickles (5 files)
- `artifacts_original/` — original pickles for reference
- `paper/figure_diagnostic.{pdf,png}` — regenerated Figure 1
- `verify_after.log` — full verify output (with PASS/FAIL details)
- `stage{3..8}.log` — per-stage training logs
- `RUN_SUMMARY.txt` — high-level summary

In [ ]:
import shutil, os, time, subprocess

# Recompute paths if globals were lost (e.g., kernel restart)
if 'RUN_DIR' not in dir():
    DRIVE_OUTPUT = '/content/drive/MyDrive/panar-x-replication'
    RUN_TAG = time.strftime('%Y%m%d_%H%M%S')
    RUN_DIR = f'{DRIVE_OUTPUT}/run_{RUN_TAG}'
    os.makedirs(RUN_DIR, exist_ok=True)
    print(f'(Re-creating run dir: {RUN_DIR})')

# Top-level summary
summary_lines = []
summary_lines.append('PANAR-X reproducibility run')
summary_lines.append(f'Run tag: {RUN_TAG}')
summary_lines.append(f'Run dir: {RUN_DIR}')
summary_lines.append(f'Finished: {time.strftime("%Y-%m-%d %H:%M:%S")}')
summary_lines.append('')

# GPU info
import torch
if torch.cuda.is_available():
    summary_lines.append(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    summary_lines.append('GPU: none (CPU run)')
summary_lines.append('')

# Verify counts
with open('/tmp/verify_after.log') as f:
    log = f.read()
pass_count = log.count('[PASS]')
fail_count = log.count('[FAIL]')
summary_lines.append(f'Verify (regenerated): PASS {pass_count}, FAIL {fail_count}')
summary_lines.append('')

with open('/tmp/RUN_SUMMARY.txt', 'w') as f:
    f.write('\n'.join(summary_lines))

# Copy directories and files to Drive
print(f'Saving to {RUN_DIR}/ ...')

shutil.copytree('artifacts', f'{RUN_DIR}/artifacts', dirs_exist_ok=True)
print('  artifacts/                         saved')

if os.path.exists('artifacts_original'):
    shutil.copytree('artifacts_original', f'{RUN_DIR}/artifacts_original', dirs_exist_ok=True)
    print('  artifacts_original/                saved')

os.makedirs(f'{RUN_DIR}/paper', exist_ok=True)
for f in ['figure_diagnostic.pdf', 'figure_diagnostic.png']:
    if os.path.exists(f'paper/{f}'):
        shutil.copy(f'paper/{f}', f'{RUN_DIR}/paper/{f}')
        print(f'  paper/{f:<28} saved')

# Logs
os.makedirs(f'{RUN_DIR}/logs', exist_ok=True)
for log_file in ['smoke_test.log', 'verify_after.log',
                 'stage3.log', 'stage4.log', 'stage5.log',
                 'stage6.log', 'stage7.log', 'stage8.log', 'RUN_SUMMARY.txt']:
    src = f'/tmp/{log_file}'
    if os.path.exists(src):
        shutil.copy(src, f'{RUN_DIR}/logs/{log_file}')
        print(f'  logs/{log_file:<26} saved')

print(f'\nAll outputs saved to: {RUN_DIR}')
print('\nFinal contents:')
subprocess.run(['ls', '-la', RUN_DIR])

## Done.

If you saw `141/141 PASS` (or close to it with epsilon-margin failures
explained by GPU non-determinism), the pipeline reproduces successfully.

The original tarball with its saved artifacts remains the canonical
submission. This notebook's outputs in Drive (`{RUN_DIR}` above) are
evidence of reproduction, not a replacement for the package.